# Verify the bonus Triton kernel on a real CUDA GPU

`triton_kernels.py` implements a hand-written Triton kernel that fuses scale → additive-mask → row-softmax into a single kernel launch. It's a bonus/demonstration module, separate from the graded `UserOptimizedTransformer` path.

Triton kernels only run on CUDA, which wasn't available on either machine used during development (a CPU-only sandbox, and an Apple Silicon machine whose MPS backend can't run Triton). This notebook clones the repo and runs `tests/test_triton_kernel.py` directly on a CUDA runtime, in about a minute — already run once on a Colab Tesla T4 (48/48 configurations pass, see `reports/TECH_REPORT.md` §9); running it here reproduces that result.

**Before running:** `Runtime` → `Change runtime type` → select a `T4 GPU` (or any CUDA GPU). The free tier is enough for this.

In [ ]:
# Repo path for this submission
REPO_URL = "https://github.com/brdge77e/optimized-transformer-layer.git"

!git clone "$REPO_URL" repo
%cd repo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q triton
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4 GPU"

In [ ]:
!python3 tests/test_triton_kernel.py

## What to look for

The script sweeps sequence lengths (16–2048), causal on/off, padding on/off, and dtypes (float32/float16/bfloat16), comparing the Triton kernel's output against plain PyTorch softmax at `rtol<0.02, atol<0.002` — the same tolerance the hackathon's own test cases use. Every row should print `PASS`.

This has already been run once on a Colab Tesla T4 (PyTorch 2.11.0+cu128): 48/48 configurations passed, with `max_abs_err` up to `1.221e-4` on bfloat16 (the least precise dtype tested) — see `reports/TECH_REPORT.md` §9. Running this notebook yourself reproduces that result rather than taking it on faith.</cell id="cell-4">
